# **LangChain Memory**

### **Objective:**  
To explore different types of memory in LangChain and understand how each type retains, summarizes, or limits conversational context. This helps in building efficient AI systems that maintain relevant context while optimizing performance and memory usage.

---

### **Note:**  
- Before running any demo, ensure that the **requirements.txt** file is installed. This file contains all the required dependencies for **all demos and guided practices under Building LLM Applications**.
- If the dependencies were already installed earlier (after creating the virtual environment), there is no need to install them again. You can directly proceed with running the demo.
- Refer to Lesson_01 **Demo_01_Zero_Shot_Prompting.ipynb** Step 1 for creating a virtual environment and installing the requirements.txt
- Ensure you select the right kernel **Python (myenv)** while running the demos
---




---


### **Step 1: Import the necessary modules**
- Import all required components for creating memory chains and managing conversational history


In [1]:
!pip install langchain-community
!pip install langchain-openai
!pip install langchain-community
!pip install langchain-core
!pip install pydantic
!pip install langchain-text-splitters
!pip install faiss-cpu
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 80.8 MB/s eta 0:00:00


In [2]:
# =====================================================
# Simplified Gemini + LangChain Memory Demo
# =====================================================

import os
import getpass
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import AIMessage
from langchain_core.runnables.history import RunnableWithMessageHistory
from pydantic import PrivateAttr


# -----------------------------
# Set Gemini API Key
# -----------------------------
os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter OpenRouter API Key: ")


# -----------------------------
# Initialize Gemini Model
# -----------------------------

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0,
    max_tokens=300,
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("placeholder", "{history}"),
    ("human", "{input}")
])


def build_chain(memory):
    return RunnableWithMessageHistory(
        prompt | llm,
        lambda session_id: memory,
        input_messages_key="input",
        history_messages_key="history",
    )


def print_memory(memory):
    for m in memory.messages:
        role = "User" if m.type == "human" else "AI"
        print(f"{role}: {m.content}")
    print()




/tmp/ipykernel_3374/1815794007.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory


Enter OpenRouter API Key: ··········


In [21]:
# print(type(memory.messages[1]))

<class 'langchain_core.messages.ai.AIMessage'>


In [22]:
for m in memory.messages:
    print(m.type)




human
ai
human
ai
human
ai


In [23]:
# =====================================================
# 1. Full Memory (Remembers Everything)
# =====================================================
print("=== Full Memory ===")

memory = ChatMessageHistory()
chain = build_chain(memory)

chain.invoke({"input": "My name is Alex"}, config={"configurable": {"session_id": "A"}})
chain.invoke({"input": "What is 2 + 2?"}, config={"configurable": {"session_id": "A"}})
chain.invoke({"input": "What is my name?"}, config={"configurable": {"session_id": "A"}})

print_memory(memory)





=== Full Memory ===


/tmp/ipykernel_3374/780495963.py:7: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  chain = build_chain(memory)


User: My name is Alex
AI: Nice to meet you, Alex! How can I assist you today?
User: What is 2 + 2?
AI: 2 + 2 equals 4.
User: What is my name?
AI: Your name is Alex.



In [4]:
memory

InMemoryChatMessageHistory(messages=[HumanMessage(content='My name is Alex', additional_kwargs={}, response_metadata={}), AIMessage(content='Nice to meet you, Alex! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 21, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 1.155e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 1.155e-05, 'upstream_inference_prompt_cost': 3.15e-06, 'upstream_inference_completions_cost': 8.4e-06}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_d0469e1700', 'id': 'gen-1782655156-2c60ZDmqFAS0MbFVoadW', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019

In [28]:
# =====================================================
# 2. Window Memory (Only Last Turn)
# =====================================================
class WindowMemory(ChatMessageHistory):
    _k: int = PrivateAttr()

    def __init__(self, k=1):
        super().__init__()
        self._k = k

    def add_message(self, message):
        super().add_message(message)
        self.messages = self.messages[-2 * self._k:]


print("=== Window Memory (k=1) ===")

memory = WindowMemory(k=1)
chain = build_chain(memory)

chain.invoke({"input": "My name is Alex"}, config={"configurable": {"session_id": "B"}})
chain.invoke({"input": "What is my name?"}, config={"configurable": {"session_id": "B"}})

print_memory(memory)




=== Window Memory (k=1) ===


/tmp/ipykernel_3374/3721861965.py:19: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  chain = build_chain(memory)


User: What is my name?
AI: Your name is Alex. How can I help you today, Alex?



In [31]:
# =====================================================
# 3. Token-Limited Memory (Simple Approximation)
# =====================================================
class TokenMemory(ChatMessageHistory):
    _max_words: int = PrivateAttr()

    def __init__(self, max_words=30):
        super().__init__()
        self._max_words = max_words

    def add_message(self, message):
        super().add_message(message)
        while sum(len(m.content.split()) for m in self.messages) > self._max_words:
            self.messages.pop(0)


print("=== Token Memory ===")

memory = TokenMemory(max_words=30)
chain = build_chain(memory)

chain.invoke({"input": "Machine Learning explained in 5 words"}, config={"configurable": {"session_id": "C"}})
chain.invoke({"input": "Neural Networks explained in 5 words"}, config={"configurable": {"session_id": "C"}})
chain.invoke({"input": "AI Assistants explained in 5 words"}, config={"configurable": {"session_id": "C"}})

print_memory(memory)




=== Token Memory ===


/tmp/ipykernel_3374/2809902338.py:20: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  chain = build_chain(memory)


AI: Data-driven algorithms for pattern recognition.
User: Neural Networks explained in 5 words
AI: Layered structures mimicking human brain.
User: AI Assistants explained in 5 words
AI: Automated tools for user support.



In [35]:
# =====================================================
# 4. Summary Memory (Compress Old Chats)
# =====================================================
class SummaryMemory(ChatMessageHistory):
    _llm: any = PrivateAttr()
    _max_turns: int = PrivateAttr()

    def __init__(self, llm, max_turns=4):
        super().__init__()
        self._llm = llm
        self._max_turns = max_turns

    def add_message(self, message):
        super().add_message(message)

        if len(self.messages) > self._max_turns:
            text = "\n".join(m.content for m in self.messages)
            summary = self._llm.invoke(f"Summarize:\n{text}").content
            self.messages = [AIMessage(content=summary)]


print("=== Summary Memory ===")

memory = SummaryMemory(llm)
chain = build_chain(memory)

chain.invoke({"input": "Hi"}, config={"configurable": {"session_id": "D"}})
print_memory(memory)
print("*"*30)
chain.invoke({"input": "What are we discussing?"}, config={"configurable": {"session_id": "D"}})
print_memory(memory)
print("*"*30)
chain.invoke({"input": "Explain further"}, config={"configurable": {"session_id": "D"}})
print_memory(memory)
print("*"*30)
chain.invoke({"input": "Give details"}, config={"configurable": {"session_id": "D"}})

print_memory(memory)

=== Summary Memory ===


/tmp/ipykernel_3374/801330254.py:25: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  chain = build_chain(memory)


User: Hi
AI: Hello! How can I assist you today?

******************************
User: Hi
AI: Hello! How can I assist you today?
User: What are we discussing?
AI: We can discuss anything you'd like! Whether you have questions, need information, or want to chat about a specific topic, just let me know what you're interested in.

******************************
AI: The conversation begins with a greeting, followed by an offer of assistance. The speaker invites the other person to discuss any topic of interest, whether it involves questions, information, or casual conversation.
AI: Sure! Here are a few topics we could discuss:

1. **General Knowledge**: Ask about history, science, technology, or any other subject you're curious about.
2. **Current Events**: I can provide information on recent news or trends up to October 2023.
3. **Advice**: If you need help with personal issues, study tips, or career advice, I can offer suggestions.
4. **Entertainment**: We can talk about movies, books, mu

In [33]:
memory

SummaryMemory(messages=[AIMessage(content='The conversation begins with a greeting, followed by an offer of assistance. The speaker invites the other person to discuss any topic of interest, whether it involves questions, information, or casual conversation.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), AIMessage(content="Sure! Here are a few topics we could discuss:\n\n1. **Current Events**: We can talk about recent news or developments in various fields like politics, science, technology, or entertainment.\n\n2. **Science and Technology**: If you're interested in advancements in technology, scientific discoveries, or how things work, we can dive into those topics.\n\n3. **Health and Wellness**: We can discuss topics related to physical and mental health, nutrition, fitness, or wellness practices.\n\n4. **Books and Literature**: If you enjoy reading, we can talk about your favorite books, authors, or literary genres.\n\n5. **History**: We can exp